# 04 · Export Power BI

In [ ]:
%pip install -q geopandas xlrd openpyxl scikit-learn requests pyarrow

In [ ]:
ROOT = "/lakehouse/default"
SEUIL_POP_DIFFUSION = 200

In [ ]:
import json
from pathlib import Path
import pandas as pd
from shapely import wkt
from shapely.geometry import mapping

OUT = Path(ROOT) / "Files/hackaton/out"
read = lambda name: spark.read.table(name).toPandas()

gold, sugg, couts = read("gold_iris"), read("gold_suggestions"), read("gold_couts")
gold = gold[gold.pop_2021 >= SEUIL_POP_DIFFUSION]

In [ ]:
gold.to_csv(OUT / "export_powerbi_iris.csv", index=False)
sugg.to_csv(OUT / "export_powerbi_suggestions.csv", index=False)
couts.to_csv(OUT / "export_powerbi_couts.csv", index=False)
props = ["ville", "code_iris", "nom_iris", "nom_commune", "pop_2021", "pop_2030_proj", "tx_annuel_15_21", "tx_proj_21_30", "evolution_predite"]
for v, g in gold.groupby("ville"):
    fc = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": r[props].to_dict(), "geometry": mapping(wkt.loads(r.geometry_wkt))} for _, r in g.iterrows()]}
    (OUT / f"iris_{v}.geojson").write_text(json.dumps(fc, default=float))
sorted(p.name for p in OUT.iterdir())